## 230968078 - Ishan Suryawanshi - Lab 7

### Exercise 1 - Tic Tac Toe using Minimax and Alpha Beta Pruning

In [2]:
nodes_minimax = 0
nodes_ab = 0

In [3]:
def initial_state():
    return [' '] * 9

In [4]:
def print_board(state):
    print()
    for row in range(3):
        cells = state[row*3 : row*3 + 3]
        print(f"  {cells[0]} | {cells[1]} | {cells[2]}")
        if row < 2:
            print("  ---------")
    print()

In [5]:
def actions(state):
    return [i for i, cell in enumerate(state) if cell == ' ']

In [6]:
def result(state, action, player_mark):
    new_state = state[:]
    new_state[action] = player_mark
    return new_state

In [7]:
def player(state):
    x_count = state.count('X')
    o_count = state.count('O')
    return 'X' if x_count == o_count else 'O'

In [8]:
def _check_winner(state, mark):
    win_lines = [
        (0, 1, 2), (3, 4, 5), (6, 7, 8),
        (0, 3, 6), (1, 4, 7), (2, 5, 8),
        (0, 4, 8), (2, 4, 6)
    ]
    return any(state[a] == state[b] == state[c] == mark for a, b, c in win_lines)

In [9]:
def terminal(state):
    return _check_winner(state, 'X') or _check_winner(state, 'O') or ' ' not in state

In [10]:
def utility(state):
    if _check_winner(state, 'X'):
        return 1
    if _check_winner(state, 'O'):
        return -1
    return 0

In [11]:
def game_status(state):
    if _check_winner(state, 'X'):
        return "X wins!"
    if _check_winner(state, 'O'):
        return "O wins!"
    if ' ' not in state:
        return "Draw!"
    return "Continue"

In [12]:
def max_value(state):
    global nodes_minimax
    nodes_minimax += 1

    if terminal(state):
        return utility(state)

    v = float('-inf')
    for action in actions(state):
        v = max(v, min_value(result(state, action, 'X')))
    return v

In [13]:
def min_value(state):
    global nodes_minimax
    nodes_minimax += 1

    if terminal(state):
        return utility(state)

    v = float('inf')
    for action in actions(state):
        v = min(v, max_value(result(state, action, 'O')))
    return v

In [14]:
def minimax_decision(state):
    best_move = None
    best_val = float('-inf')

    for action in actions(state):
        val = min_value(result(state, action, 'X'))
        if val > best_val:
            best_val = val
            best_move = action

    return best_move, best_val

In [15]:
def max_value_ab(state, alpha, beta):
    global nodes_ab
    nodes_ab += 1

    if terminal(state):
        return utility(state)

    v = float('-inf')
    for action in actions(state):
        v = max(v, min_value_ab(result(state, action, 'X'), alpha, beta))
        if v >= beta:          # Beta cut-off
            return v
        alpha = max(alpha, v)
    return v

In [16]:
def min_value_ab(state, alpha, beta):
    global nodes_ab
    nodes_ab += 1

    if terminal(state):
        return utility(state)

    v = float('inf')
    for action in actions(state):
        v = min(v, max_value_ab(result(state, action, 'O'), alpha, beta))
        if v <= alpha:         # Alpha cut-off
            return v
        beta = min(beta, v)
    return v

In [17]:
def alphabeta_decision(state):
    best_move = None
    best_val = float('-inf')
    alpha = float('-inf')
    beta = float('inf')

    for action in actions(state):
        val = min_value_ab(result(state, action, 'X'), alpha, beta)
        if val > best_val:
            best_val = val
            best_move = action
        alpha = max(alpha, best_val)

    return best_move, best_val

In [18]:
def index_to_rowcol(index):
    return (index // 3 + 1, index % 3 + 1)

In [19]:
def human_move(state):
    available = actions(state)
    print(f"  Available positions: {[i+1 for i in available]}  (enter 1-9)")

    while True:
        try:
            choice = int(input("  Your move (O): ")) - 1
            if choice in available:
                return choice
            else:
                print("  Invalid position. Cell is already taken or out of range.")
        except ValueError:
            print("  Please enter a number between 1 and 9.")

In [20]:
def play_game():
    global nodes_minimax, nodes_ab

    print("=" * 50)
    print("   TIC-TAC-TOE  —  Human (O) vs AI (X)")
    print("=" * 50)
    print("Board positions:")
    print("   1 | 2 | 3")
    print("   ---------")
    print("   4 | 5 | 6")
    print("   ---------")
    print("   7 | 8 | 9")

    print("\nSelect AI algorithm:")
    print("  1. Minimax")
    print("  2. Alpha-Beta Pruning")
    while True:
        algo_choice = input("Enter choice (1 or 2): ").strip()
        if algo_choice in ('1', '2'):
            break
        print("  Invalid choice. Enter 1 or 2.")

    use_alphabeta = (algo_choice == '2')
    algo_name = "Alpha-Beta Pruning" if use_alphabeta else "Minimax"
    print(f"\nAI will use: {algo_name}\n")

    nodes_minimax = 0
    nodes_ab = 0

    state = initial_state()
    print_board(state)

    while not terminal(state):
        current = player(state)

        if current == 'X':
            print("AI (X) is thinking...")
            if use_alphabeta:
                move, val = alphabeta_decision(state)
            else:
                move, val = minimax_decision(state)

            state = result(state, move, 'X')
            row, col = index_to_rowcol(move)
            print(f"  AI chose position {move+1}  →  (row {row}, col {col})")
            print_board(state)
            status = game_status(state)
            print(f"  Status: {status}")
            print()

        else:
            print("Your turn (O):")
            move = human_move(state)
            state = result(state, move, 'O')
            print_board(state)
            status = game_status(state)
            print(f"  Status: {status}")
            print()

    print("=" * 50)
    print("           GAME OVER")
    print("=" * 50)
    nodes_minimax = 0
    nodes_ab = 0
    minimax_decision(initial_state())
    mm_count = nodes_minimax

    nodes_minimax = 0
    nodes_ab = 0
    alphabeta_decision(initial_state())
    ab_count = nodes_ab

    print(f"\n  Nodes evaluated using Minimax:     {mm_count}")
    print(f"  Nodes evaluated using Alpha-Beta:  {ab_count}")
    reduction = (1 - ab_count / mm_count) * 100 if mm_count else 0
    print(f"  Node reduction by Alpha-Beta:      {reduction:.1f}%")

    print("\n  WHY ALPHA-BETA EVALUATES FEWER NODES:")
    print("  Alpha-Beta pruning eliminates branches that cannot possibly affect")
    print("  the final decision because a better option has already been found.")
    print("  When β ≤ α, further exploration of that subtree is skipped entirely,")
    print("  reducing the search space without changing the optimal result.")
    print()

In [21]:
play_game()

   TIC-TAC-TOE  —  Human (O) vs AI (X)
Board positions:
   1 | 2 | 3
   ---------
   4 | 5 | 6
   ---------
   7 | 8 | 9

Select AI algorithm:
  1. Minimax
  2. Alpha-Beta Pruning
Enter choice (1 or 2): 1

AI will use: Minimax


    |   |  
  ---------
    |   |  
  ---------
    |   |  

AI (X) is thinking...
  AI chose position 1  →  (row 1, col 1)

  X |   |  
  ---------
    |   |  
  ---------
    |   |  

  Status: Continue

Your turn (O):
  Available positions: [2, 3, 4, 5, 6, 7, 8, 9]  (enter 1-9)
  Your move (O): 5

  X |   |  
  ---------
    | O |  
  ---------
    |   |  

  Status: Continue

AI (X) is thinking...
  AI chose position 2  →  (row 1, col 2)

  X | X |  
  ---------
    | O |  
  ---------
    |   |  

  Status: Continue

Your turn (O):
  Available positions: [3, 4, 6, 7, 8, 9]  (enter 1-9)
  Your move (O): 3

  X | X | O
  ---------
    | O |  
  ---------
    |   |  

  Status: Continue

AI (X) is thinking...
  AI chose position 7  →  (row 3, col 1)

  X | X 

In [22]:
play_game()

   TIC-TAC-TOE  —  Human (O) vs AI (X)
Board positions:
   1 | 2 | 3
   ---------
   4 | 5 | 6
   ---------
   7 | 8 | 9

Select AI algorithm:
  1. Minimax
  2. Alpha-Beta Pruning
Enter choice (1 or 2): 2

AI will use: Alpha-Beta Pruning


    |   |  
  ---------
    |   |  
  ---------
    |   |  

AI (X) is thinking...
  AI chose position 1  →  (row 1, col 1)

  X |   |  
  ---------
    |   |  
  ---------
    |   |  

  Status: Continue

Your turn (O):
  Available positions: [2, 3, 4, 5, 6, 7, 8, 9]  (enter 1-9)
  Your move (O): 3

  X |   | O
  ---------
    |   |  
  ---------
    |   |  

  Status: Continue

AI (X) is thinking...
  AI chose position 4  →  (row 2, col 1)

  X |   | O
  ---------
  X |   |  
  ---------
    |   |  

  Status: Continue

Your turn (O):
  Available positions: [2, 5, 6, 7, 8, 9]  (enter 1-9)
  Your move (O): 7

  X |   | O
  ---------
  X |   |  
  ---------
  O |   |  

  Status: Continue

AI (X) is thinking...
  AI chose position 5  →  (row 2, col 2

### Ex2 - Car Overtaking Game With MiniMax and Alpha Beta Pruning

In [23]:
ROAD_SIZE = 7
GOAL      = ROAD_SIZE - 1

In [24]:
def initial_state():
    return (0, 1, True)

In [25]:
def actions(state):
    x, o, max_turn = state
    pos = x if max_turn else o
    opp = o if max_turn else x
    return [step for step in (1, 2) if pos + step <= GOAL and pos + step != opp]

In [26]:
def result(state, move):
    x, o, max_turn = state
    if max_turn:
        return (x + move, o, False)
    else:
        return (x, o + move, True)

In [27]:
def terminal(state):
    x, o, _ = state
    return x == GOAL or o == GOAL

In [28]:
def utility(state):
    x, o, _ = state
    if x == GOAL: return +1
    if o == GOAL: return -1
    return 0

In [29]:
def print_road(state):
    x, o, max_turn = state
    road = ['_'] * ROAD_SIZE
    road[x] = 'X'
    road[o] = 'O'
    cells = ' | '.join(f"{road[i]}({i})" for i in range(ROAD_SIZE))
    print(f"  Road : [ {cells} ]")
    print(f"  Turn : {'MAX (X)' if max_turn else 'MIN (O)'}")

In [30]:
def max_value(state):
    global nodes_minimax
    nodes_minimax += 1
    if terminal(state):
        return utility(state)
    v = float('-inf')
    for move in actions(state):
        v = max(v, min_value(result(state, move)))
    return v

In [31]:
def min_value(state):
    global nodes_minimax
    nodes_minimax += 1
    if terminal(state):
        return utility(state)
    v = float('inf')
    for move in actions(state):
        v = min(v, max_value(result(state, move)))
    return v

In [32]:
def minimax_decision(state):
    best_move, best_val = None, float('-inf')
    for move in actions(state):
        val = min_value(result(state, move))
        if val > best_val:
            best_val, best_move = val, move
    return best_move, best_val

In [33]:
def max_value_ab(state, alpha, beta):
    global nodes_ab
    nodes_ab += 1
    if terminal(state):
        return utility(state)
    v = float('-inf')
    for move in actions(state):
        v = max(v, min_value_ab(result(state, move), alpha, beta))
        if v >= beta:
            return v
        alpha = max(alpha, v)
    return v

In [34]:
def min_value_ab(state, alpha, beta):
    global nodes_ab
    nodes_ab += 1
    if terminal(state):
        return utility(state)
    v = float('inf')
    for move in actions(state):
        v = min(v, max_value_ab(result(state, move), alpha, beta))
        if v <= alpha:
            return v
        beta = min(beta, v)
    return v

In [35]:
def alphabeta_decision(state):
    best_move, best_val = None, float('-inf')
    alpha, beta = float('-inf'), float('inf')
    for move in actions(state):
        val = min_value_ab(result(state, move), alpha, beta)
        if val > best_val:
            best_val, best_move = val, move
        alpha = max(alpha, best_val)
    return best_move, best_val

In [36]:
def play_game(algorithm='alphabeta'):
    global nodes_minimax, nodes_ab
    nodes_minimax = 0
    nodes_ab = 0

    _init = initial_state()
    max_value(_init)
    _mm_total = nodes_minimax

    nodes_ab = 0
    max_value_ab(_init, float('-inf'), float('inf'))
    _ab_total = nodes_ab

    decide = alphabeta_decision if algorithm == 'alphabeta' else minimax_decision

    print('=' * 54)
    print('   CAR OVERTAKING GAME  —  AI vs AI')
    print(f'   Algorithm : {"Alpha-Beta Pruning" if algorithm == "alphabeta" else "Minimax"}')
    print('=' * 54)
    print(f'  Road : {ROAD_SIZE} cells (0 → {GOAL})  |  X@0, O@1, MAX moves first')
    print()

    state   = initial_state()
    turn_no = 0

    while not terminal(state):
        turn_no += 1
        _, _, max_turn = state
        print(f'Turn {turn_no} — {"MAX (X)" if max_turn else "MIN (O)"}')
        print_road(state)

        move, val = decide(state)
        state = result(state, move)
        nx, no, _ = state
        print(f'  Move  : +{move} cell(s)  →  X@{nx}, O@{no}  (value: {val:+d})')
        print()

    print('─' * 54)
    print_road(state)
    winner = 'X wins! (MAX)' if state[0] == GOAL else 'O wins! (MIN)'
    print(f'\n  *** GAME OVER — {winner} ***\n')

    reduction = (_mm_total - _ab_total) / _mm_total * 100 if _mm_total else 0
    print('=' * 54)
    print('  NODE COUNT COMPARISON')
    print('=' * 54)
    print(f'  Minimax nodes    : {_mm_total:>6}')
    print(f'  Alpha-Beta nodes : {_ab_total:>6}  ({reduction:.1f}% reduction)')
    print()
    print('  WHY ALPHA-BETA EVALUATES FEWER NODES:')
    print('  When β ≤ α, further search cannot change the decision,')
    print('  so that subtree is skipped. Final move = same as Minimax.')
    print()

In [37]:
play_game('minimax')

   CAR OVERTAKING GAME  —  AI vs AI
   Algorithm : Minimax
  Road : 7 cells (0 → 6)  |  X@0, O@1, MAX moves first

Turn 1 — MAX (X)
  Road : [ X(0) | O(1) | _(2) | _(3) | _(4) | _(5) | _(6) ]
  Turn : MAX (X)
  Move  : +2 cell(s)  →  X@2, O@1  (value: +1)

Turn 2 — MIN (O)
  Road : [ _(0) | O(1) | X(2) | _(3) | _(4) | _(5) | _(6) ]
  Turn : MIN (O)
  Move  : +2 cell(s)  →  X@2, O@3  (value: +1)

Turn 3 — MAX (X)
  Road : [ _(0) | _(1) | X(2) | O(3) | _(4) | _(5) | _(6) ]
  Turn : MAX (X)
  Move  : +2 cell(s)  →  X@4, O@3  (value: +1)

Turn 4 — MIN (O)
  Road : [ _(0) | _(1) | _(2) | O(3) | X(4) | _(5) | _(6) ]
  Turn : MIN (O)
  Move  : +2 cell(s)  →  X@4, O@5  (value: +1)

Turn 5 — MAX (X)
  Road : [ _(0) | _(1) | _(2) | _(3) | X(4) | O(5) | _(6) ]
  Turn : MAX (X)
  Move  : +2 cell(s)  →  X@6, O@5  (value: +1)

──────────────────────────────────────────────────────
  Road : [ _(0) | _(1) | _(2) | _(3) | _(4) | O(5) | X(6) ]
  Turn : MIN (O)

  *** GAME OVER — X wins! (MAX) ***

  NOD

In [38]:
play_game('alphabeta')

   CAR OVERTAKING GAME  —  AI vs AI
   Algorithm : Alpha-Beta Pruning
  Road : 7 cells (0 → 6)  |  X@0, O@1, MAX moves first

Turn 1 — MAX (X)
  Road : [ X(0) | O(1) | _(2) | _(3) | _(4) | _(5) | _(6) ]
  Turn : MAX (X)
  Move  : +2 cell(s)  →  X@2, O@1  (value: +1)

Turn 2 — MIN (O)
  Road : [ _(0) | O(1) | X(2) | _(3) | _(4) | _(5) | _(6) ]
  Turn : MIN (O)
  Move  : +2 cell(s)  →  X@2, O@3  (value: +1)

Turn 3 — MAX (X)
  Road : [ _(0) | _(1) | X(2) | O(3) | _(4) | _(5) | _(6) ]
  Turn : MAX (X)
  Move  : +2 cell(s)  →  X@4, O@3  (value: +1)

Turn 4 — MIN (O)
  Road : [ _(0) | _(1) | _(2) | O(3) | X(4) | _(5) | _(6) ]
  Turn : MIN (O)
  Move  : +2 cell(s)  →  X@4, O@5  (value: +1)

Turn 5 — MAX (X)
  Road : [ _(0) | _(1) | _(2) | _(3) | X(4) | O(5) | _(6) ]
  Turn : MAX (X)
  Move  : +2 cell(s)  →  X@6, O@5  (value: +1)

──────────────────────────────────────────────────────
  Road : [ _(0) | _(1) | _(2) | _(3) | _(4) | O(5) | X(6) ]
  Turn : MIN (O)

  *** GAME OVER — X wins! (MAX)

In [39]:
def heuristic(state):
    x, o, _ = state
    return (GOAL - o) - (GOAL - x)

In [40]:
def play_game_extended(road_size=10, obstacles=None, depth_limit=4):
    global ROAD_SIZE, GOAL
    ROAD_SIZE = road_size
    GOAL      = road_size - 1
    obs       = set(obstacles or [])

    def actions_ext(state):
        x, o, max_turn = state
        pos = x if max_turn else o
        opp = o if max_turn else x
        return [
            step for step in (1, 2)
            if pos + step <= GOAL
            and pos + step != opp
            and pos + step not in obs
        ]

    def dl_max(state, depth, alpha, beta):
        if terminal(state): return utility(state)
        if depth == 0:      return heuristic(state)
        v = float('-inf')
        for move in actions_ext(state):
            v = max(v, dl_min(result(state, move), depth - 1, alpha, beta))
            if v >= beta: return v
            alpha = max(alpha, v)
        return v

    def dl_min(state, depth, alpha, beta):
        if terminal(state): return utility(state)
        if depth == 0:      return heuristic(state)
        v = float('inf')
        for move in actions_ext(state):
            v = min(v, dl_max(result(state, move), depth - 1, alpha, beta))
            if v <= alpha: return v
            beta = min(beta, v)
        return v

    def decide_ext(state):
        best_move, best_val = None, float('-inf')
        a, b = float('-inf'), float('inf')
        for move in actions_ext(state):
            val = dl_min(result(state, move), depth_limit - 1, a, b)
            if val > best_val:
                best_val, best_move = val, move
            a = max(a, best_val)
        return best_move, best_val

    print('=' * 54)
    print('   EXTENDED CAR OVERTAKING GAME')
    print(f'   Road size   : {road_size} cells  (0 → {GOAL})')
    print(f'   Obstacles   : {sorted(obs) if obs else "none"}')
    print(f'   Depth limit : {depth_limit}')
    print('=' * 54)

    state   = initial_state()
    turn_no = 0

    while not terminal(state):    # <-- calling terminal() defined above
        turn_no += 1
        _, _, max_turn = state
        print(f'Turn {turn_no} — {"MAX (X)" if max_turn else "MIN (O)"}')
        print_road(state)         # <-- calling print_road() defined above

        move, val = decide_ext(state)
        if move is None:
            print('  No valid moves — game stalled.')
            break
        state = result(state, move)   # <-- calling result() defined above
        nx, no, _ = state
        print(f'  Move  : +{move}  →  X@{nx}, O@{no}  (h/utility: {val:+})')
        print()

    if terminal(state):           # <-- calling terminal() defined above
        winner = 'X wins! (MAX)' if state[0] == GOAL else 'O wins! (MIN)'
        print(f'  *** GAME OVER — {winner} ***')
    print()

    # Reset globals to default
    ROAD_SIZE = 7
    GOAL      = 6

In [41]:
play_game_extended()

   EXTENDED CAR OVERTAKING GAME
   Road size   : 10 cells  (0 → 9)
   Obstacles   : none
   Depth limit : 4
Turn 1 — MAX (X)
  Road : [ X(0) | O(1) | _(2) | _(3) | _(4) | _(5) | _(6) | _(7) | _(8) | _(9) ]
  Turn : MAX (X)
  Move  : +2  →  X@2, O@1  (h/utility: -1)

Turn 2 — MIN (O)
  Road : [ _(0) | O(1) | X(2) | _(3) | _(4) | _(5) | _(6) | _(7) | _(8) | _(9) ]
  Turn : MIN (O)
  Move  : +2  →  X@2, O@3  (h/utility: +1)

Turn 3 — MAX (X)
  Road : [ _(0) | _(1) | X(2) | O(3) | _(4) | _(5) | _(6) | _(7) | _(8) | _(9) ]
  Turn : MAX (X)
  Move  : +2  →  X@4, O@3  (h/utility: -1)

Turn 4 — MIN (O)
  Road : [ _(0) | _(1) | _(2) | O(3) | X(4) | _(5) | _(6) | _(7) | _(8) | _(9) ]
  Turn : MIN (O)
  Move  : +2  →  X@4, O@5  (h/utility: +1)

Turn 5 — MAX (X)
  Road : [ _(0) | _(1) | _(2) | _(3) | X(4) | O(5) | _(6) | _(7) | _(8) | _(9) ]
  Turn : MAX (X)
  Move  : +2  →  X@6, O@5  (h/utility: -1)

Turn 6 — MIN (O)
  Road : [ _(0) | _(1) | _(2) | _(3) | _(4) | O(5) | X(6) | _(7) | _(8) | _(9) ]